# Fine-tuning con Hard Negatives (Celulares)

Este notebook entrena el modelo agregando 28 imágenes de celulares de costado como **hard negatives** para reducir falsos positivos de `knife`.

**Estrategia:**
1. Cargar dataset aumentado desde Drive
2. Subir 28 celulares desde local (VSCode)
3. Integrarlos al training (NO al test)
4. Fine-tuning desde checkpoint existente

In [ ]:
# Instalar dependencias
!pip install -q torchmetrics

In [ ]:
# Montar Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Ir al repositorio en Drive
%cd /content/drive/MyDrive/procesamiento-imagenes

In [ ]:
# Copiar dataset aumentado a disco local (más rápido para GPU)
!cp "/content/drive/MyDrive/procesamiento-imagenes/dataset_augmented.zip" /content/dataset_augmented.zip
!unzip -q /content/dataset_augmented.zip -d /content/dataset_local

## Subir Hard Negatives (Celulares)

**IMPORTANTE:** Ejecutá esto desde VSCode. Los archivos se suben desde tu máquina local.

Ruta local: `/home/gbenito/universidad/procesamiento-imagenes-unlu/data/hard_negatives_celulares/`

In [ ]:
# Crear carpeta para hard negatives en Colab
!mkdir -p /content/hard_negatives/images
!mkdir -p /content/hard_negatives/xmls

In [ ]:
# Subir archivos desde VSCode (clic en carpeta o drag & drop)
# O usar este código para upload programático:
from google.colab import files
import os

print("📤 Subiendo imágenes de celulares...")
# Opcional: comentá esto si ya subiste manualmente
uploaded_imgs = files.upload()
for filename in uploaded_imgs:
    !mv "{filename}" /content/hard_negatives/images/

print("\n📤 Subiendo XMLs de celulares...")
uploaded_xmls = files.upload()
for filename in uploaded_xmls:
    !mv "{filename}" /content/hard_negatives/xmls/

In [ ]:
# Verificar que se subieron los 28
!echo "Imágenes:"
!ls /content/hard_negatives/images | wc -l
!echo "XMLs:"
!ls /content/hard_negatives/xmls | wc -l

## Integrar Hard Negatives al Training

Los copiamos al dataset de training (NO al test) para que el modelo aprenda que "celular ≠ cuchillo".

In [ ]:
# Copiar al dataset de training
!cp /content/hard_negatives/images/* /content/dataset_local/dataset_augmented/images/
!cp /content/hard_negatives/xmls/* /content/dataset_local/dataset_augmented/xmls/

print("✅ Hard negatives integrados al training")

In [ ]:
# Verificar cantidad total ahora
!echo "Total imágenes training:"
!ls /content/dataset_local/dataset_augmented/images | wc -l
!echo "Total XMLs training:"
!ls /content/dataset_local/dataset_augmented/xmls | wc -l

## Fine-tuning

Ajustamos hiperparámetros para fine-tuning:
- **LR bajo:** `1e-5` para no destruir conocimiento previo
- **Pocas épocas:** 10-15 con early stopping
- **Batch pequeño:** 4-8 para convergencia suave
- **Desde checkpoint:** continuamos desde el mejor modelo anterior

In [ ]:
# Fine-tuning desde checkpoint anterior
# AJUSTÁ la ruta del checkpoint según tu modelo actual
!python3 pipeline_entrenamiento.py \
  --skip-stages split augment \
  --resume /content/drive/MyDrive/procesamiento-imagenes/results_standard/best_model.pth \
  --augmented-images /content/dataset_local/dataset_augmented/images \
  --augmented-xmls /content/dataset_local/dataset_augmented/xmls \
  --epochs 15 \
  --batch-size 6 \
  --lr 1e-5 \
  --enhance \
  --amp \
  --save-every 5 \
  --patience 5 \
  --output-dir results_finetuning_negatives

## Evaluación

Después del fine-tuning, verificá:
1. **FP de knife bajaron?** (métrica clave)
2. **Recall de knife se mantuvo?** (no debe bajar más de 2-3%)
3. **Matriz de confusión:** verificar que casos "celular→knife" bajaron

In [ ]:
# Ver resultados
!ls -lh results_finetuning_negatives/

In [ ]:
# Copiar resultados a Drive
!cp -r results_finetuning_negatives /content/drive/MyDrive/procesamiento-imagenes/
print("✅ Resultados guardados en Drive")

## Prueba Manual

Probá el nuevo modelo con un video de celular para ver si bajaron los FPs.

In [ ]:
# Ejemplo de inferencia con el nuevo modelo
# (ajustá según tu pipeline de inferencia)
# !python3 src/weapon_detection/inference/run_pipeline.py \
#   --input test_celular.mp4 \
#   --weapon-model results_finetuning_negatives/best_model.pth